# 00 — Setup & Concepts

**Goal:** Understand *how* moto works before touching any AWS API.

## What is moto?

moto intercepts every HTTP call that `boto3` makes to AWS and returns a fake (but realistic) response — no real AWS account needed. Think of it as a local in-memory AWS.

Key facts:
- The mock is **NOT** globally active. You must turn it on explicitly.
- All mock state is **ephemeral** — it resets when the mock context ends.
- moto supports 200+ AWS services. We use: S3, Glue, Athena, EC2.
- Limitations to know up-front:
  - Athena does **not** execute SQL (it returns mock results you inject)
  - Glue crawlers do **not** scan S3
  - EC2 instances start instantly in `running` state
  - IAM permissions are **not** enforced by default

## 3 Ways to Activate a moto Mock

### 1. Decorator (best for pytest test functions)

In [1]:
import boto3
from moto import mock_aws

# The decorator activates the mock for the entire function body.
@mock_aws
def demo_decorator():
    s3 = boto3.client("s3", region_name="us-east-1")
    s3.create_bucket(Bucket="demo-bucket")
    buckets = s3.list_buckets()["Buckets"]
    print("Buckets (decorator):", [b["Name"] for b in buckets])

demo_decorator()

Buckets (decorator): ['demo-bucket']


### 2. Context Manager (best for notebook cells — recommended for this tutorial)

In [2]:
with mock_aws():
    s3 = boto3.client("s3", region_name="us-east-1")
    s3.create_bucket(Bucket="demo-bucket")
    buckets = s3.list_buckets()["Buckets"]
    print("Buckets (context manager):", [b["Name"] for b in buckets])

# Outside the `with` block the mock is gone — the next call would hit real AWS.
print("Mock ended. State is cleared.")

Buckets (context manager): ['demo-bucket']
Mock ended. State is cleared.


### 3. Class-level setUp/tearDown (best for unittest.TestCase)

In [3]:
import unittest

class MyTest(unittest.TestCase):
    def setUp(self):
        self.mock = mock_aws()
        self.mock.start()  # activate mock
        self.s3 = boto3.client("s3", region_name="us-east-1")

    def tearDown(self):
        self.mock.stop()  # deactivate mock — state is cleared

    def test_create_bucket(self):
        self.s3.create_bucket(Bucket="unittest-bucket")
        buckets = self.s3.list_buckets()["Buckets"]
        self.assertEqual(len(buckets), 1)
        print("TestCase-style mock: OK")

suite = unittest.TestLoader().loadTestsFromTestCase(MyTest)
unittest.TextTestRunner(verbosity=2).run(suite)

test_create_bucket (__main__.MyTest.test_create_bucket) ... 

ok


----------------------------------------------------------------------
Ran 1 test in 0.045s

OK


TestCase-style mock: OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Credential Setup — Why Dummy Credentials Are Required

boto3 validates that credentials exist **before** making any call, even inside a moto mock. If there are no real `~/.aws/credentials` on the machine, you get a `NoCredentialsError` *before* moto can intercept the request.

**Fix:** Set fake env vars once at the top of your session.

In [4]:
import os

# Set these once — they persist for the whole kernel session.
os.environ["AWS_ACCESS_KEY_ID"] = "testing"
os.environ["AWS_SECRET_ACCESS_KEY"] = "testing"
os.environ["AWS_SECURITY_TOKEN"] = "testing"
os.environ["AWS_SESSION_TOKEN"] = "testing"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

# OR: import helpers — it sets these automatically.
import sys
sys.path.insert(0, "..")
import helpers  # sets env vars as a side effect

print("Credentials configured:", os.environ["AWS_ACCESS_KEY_ID"])

Credentials configured: testing


## boto3 Client vs Resource

| | `client` | `resource` |
|---|---|---|
| Returns | raw dict (JSON) | Python objects with methods |
| Maps to | 1-to-1 with AWS API docs | Higher-level abstraction |
| Interview preference | **Use this** | Avoid — harder to reason about |

**Rule:** In interviews, always use `boto3.client(...)`. The AWS docs show client API call names directly.

In [5]:
with mock_aws():
    # client — low-level, returns dicts
    s3_client = boto3.client("s3", region_name="us-east-1")
    s3_client.create_bucket(Bucket="client-bucket")
    resp = s3_client.list_buckets()
    print("client response type:", type(resp))           # dict
    print("bucket names:", [b["Name"] for b in resp["Buckets"]])

    # resource — high-level, returns objects (less common in interviews)
    s3_resource = boto3.resource("s3", region_name="us-east-1")
    bucket = s3_resource.Bucket("client-bucket")         # object, not dict
    print("resource type:", type(bucket))

client response type: <class 'dict'>
bucket names: ['client-bucket']
resource type: <class 'boto3.resources.factory.s3.Bucket'>


## Reading a boto3 Response Dict

Every boto3 `client` response has the same top-level shape:
```
{
  "ResponseMetadata": { "HTTPStatusCode": 200, ... },
  <actual payload keys>: ...
}
```
Always check `ResponseMetadata.HTTPStatusCode` to know if the call succeeded.

In [6]:
import json

with mock_aws():
    s3 = boto3.client("s3", region_name="us-east-1")
    s3.create_bucket(Bucket="my-bucket")
    s3.put_object(Bucket="my-bucket", Key="hello.txt", Body=b"world")
    
    resp = s3.get_object(Bucket="my-bucket", Key="hello.txt")
    print("HTTP status:", resp["ResponseMetadata"]["HTTPStatusCode"])  # 200
    print("Content-Type:", resp.get("ContentType", "binary/octet-stream"))
    
    # Body is a StreamingBody — you MUST call .read() to get bytes
    body_bytes = resp["Body"].read()
    print("Body bytes:", body_bytes)          # b'world'
    print("Body decoded:", body_bytes.decode())  # 'world'

HTTP status: 200
Content-Type: binary/octet-stream
Body bytes: b'world'
Body decoded: world


## Walk-through: `main.py`

Let's read the existing file and explain every line.

In [7]:
# Read and display main.py
with open("../main.py") as f:
    print(f.read())

import boto3
from moto import mock_aws, ThreadedMotoServer

class MyModel:
    def __init__(self, name, value):
        self.name = name
        self.value = value

    def save(self):
        s3 = boto3.client("s3", region_name="us-east-1")
        s3.put_object(Bucket="mybucket", Key=self.name, Body=self.value)


@mock_aws
def test_my_model_save():
    mock = mock_aws()
    mock.start()

    conn = boto3.resource("s3", region_name="us_east-1")
    # Create the bucket since everything is in Moto's Virtual AWS
    conn.create_bucket(Bucket="mybucket")

    model_instance = MyModel("tanat", "metmaolee")
    model_instance.save()

    body = conn.Object("mybucket", "tanat").get()[
        "Body"].read().decode("utf-8")
    
    assert body == "metmaolee"

    mock.stop()


def main():
    print("Hello from moto-boto3!")


if __name__ == "__main__":
    main()



**Line-by-line explanation:**

1. `@mock_aws` — activates the mock for the entire function; any boto3 call inside goes to moto, not AWS.
2. `conn = boto3.client("s3", ...)` — creates the S3 client; must be done *inside* the mock.
3. `conn.create_bucket(Bucket=...)` — creates an in-memory bucket.
4. `MyModel.save()` — calls `put_object`, which stores data in the in-memory bucket.
5. `get_object(...)["Body"].read()` — reads the data back. Note the `.read()` on `Body`.
6. `assert body == b"metmaolee"` — confirms round-trip correctness.

This is the **core pattern** you will repeat for every AWS service in this tutorial.

## Summary Cheat Sheet

| Topic | Key Point |
|---|---|
| Activate mock | `@mock_aws`, `with mock_aws():`, or `.start()/.stop()` |
| Credentials | Always set dummy env vars or use `helpers.make_boto3_client()` |
| Client vs resource | Always use `client` in interviews |
| Response shape | `resp["ResponseMetadata"]["HTTPStatusCode"]` + payload keys |
| StreamingBody | Always call `.read()` on `resp["Body"]` |
| Athena in moto | Doesn't execute SQL — use result injection |
| Glue crawlers | Don't scan S3 — register tables manually |
| EC2 state | Instances are instantly `running` — no delay |

**Next notebook:** [01_s3_data_lake_foundations.ipynb](01_s3_data_lake_foundations.ipynb)